<div style="font-family: Arial; font-size: 12pt; line-height: 1.5;">
<center>
<h2>Universidad Autónoma de Aguascalientes</h2>
<h3>Departamento: Ciencias de la Computación</h3>
<h3>Carrera: Ingeniería en Computación Inteligente</h3>
<h3>Curso: Machine y Deep Learning</h3>
<h3>Maestro: Dr. Francisco Javier Luna Rosas</h3>
<h3>Alumno: Guillermo González Lara (237864)</h3>
<h3>Semestre: Enero - Junio 2026</h3>
<br>
<h1>Fase 1 — Reconocimiento de Patrones: Firmas en Cheques Off-Line</h1>
</center>
</div>

**Descripción general**
Implementación de un sistema de aprendizaje supervisado para el reconocimiento de firmas *off-line*. El modelo entrena desde cero una red U-Net para la segmentación de la firma, seguida de una extracción de características morfológicas (cuadrícula) y clasificación final (+5 auténtica, -5 falsificada) usando NNBP, SVM, KNN y Naive Bayes.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import warnings
warnings.filterwarnings('ignore')

# Rutas del dataset
DATASET_ROOT = './content/sample_data/dataset/'
REAL_DIR = os.path.join(DATASET_ROOT, 'real')
FAKE_DIR = os.path.join(DATASET_ROOT, 'fake')
TEST_REAL_PATH = os.path.join(DATASET_ROOT, 'real_test.jpg')
TEST_FAKE_PATH = os.path.join(DATASET_ROOT, 'fake_test.jpg')

IMG_SIZE = 256

In [ ]:
def generar_mascara_automatica(img_gray):
    """
    Genera una máscara de la firma usando binarización si no se cuenta con máscaras manuales.
    Esto servirá como 'ground truth' para entrenar la U-Net desde cero.
    """
    # Suavizado y binarización de Otsu invertida (trazo blanco, fondo negro)
    blur = cv2.GaussianBlur(img_gray, (5,5), 0)
    _, mask = cv2.threshold(blur, 0, 1, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return mask

def cargar_y_preparar_datos(carpeta, label):
    imagenes_X = []
    mascaras_Y = []
    labels_Z = []
    
    if not os.path.exists(carpeta):
        print(f"Carpeta no encontrada: {carpeta}")
        return imagenes_X, mascaras_Y, labels_Z
        
    for filename in os.listdir(carpeta):
        img_path = os.path.join(carpeta, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        if img is not None:
            # 1. Imagen Original (Entrada X para U-Net)
            img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img_norm = img_resized / 255.0
            
            # 2. Máscara Objetivo (Salida Y para U-Net)
            mask = generar_mascara_automatica(img_resized)
            
            imagenes_X.append(img_norm)
            mascaras_Y.append(mask)
            labels_Z.append(label)
            
    return imagenes_X, mascaras_Y, labels_Z

# Cargar dataset
print("Cargando imágenes y generando máscaras de entrenamiento...")
real_X, real_Y, real_labels = cargar_y_preparar_datos(REAL_DIR, 5)
fake_X, fake_Y, fake_labels = cargar_y_preparar_datos(FAKE_DIR, -5)

X_unet = np.array(real_X + fake_X).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
Y_unet = np.array(real_Y + fake_Y).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
y_target = np.array(real_labels + fake_labels)

print(f"Total de muestras preparadas: {len(X_unet)}")

# Visualizar un ejemplo del Ground Truth generado
if len(X_unet) > 0:
    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(X_unet[0].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')
    plt.title("Entrada a U-Net (X)")
    plt.subplot(1, 2, 2)
    plt.imshow(Y_unet[0].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')
    plt.title("Target de U-Net (Y)")
    plt.show()

In [ ]:
def build_unet(input_shape=(256, 256, 1)):
    inputs = Input(input_shape)
    
    # Reducimos un poco los filtros para evitar sobreajuste extremo en un dataset de 30 imágenes
    conv1 = Conv2D(16, 3, activation='relu', padding='same')(inputs)
    conv1 = Conv2D(16, 3, activation='relu', padding='same')(conv1)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)

    conv2 = Conv2D(32, 3, activation='relu', padding='same')(pool1)
    conv2 = Conv2D(32, 3, activation='relu', padding='same')(conv2)
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)

    conv3 = Conv2D(64, 3, activation='relu', padding='same')(pool2)
    conv3 = Conv2D(64, 3, activation='relu', padding='same')(conv3)

    up4 = concatenate([UpSampling2D(size=(2, 2))(conv3), conv2], axis=-1)
    conv4 = Conv2D(32, 3, activation='relu', padding='same')(up4)
    conv4 = Conv2D(32, 3, activation='relu', padding='same')(conv4)

    up5 = concatenate([UpSampling2D(size=(2, 2))(conv4), conv1], axis=-1)
    conv5 = Conv2D(16, 3, activation='relu', padding='same')(up5)
    conv5 = Conv2D(16, 3, activation='relu', padding='same')(conv5)

    outputs = Conv2D(1, 1, activation='sigmoid')(conv5)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

unet_model = build_unet()

if len(X_unet) > 0:
    print("Iniciando entrenamiento de U-Net desde cero...")
    # Entrenamos la red con el dataset nuevo
    history = unet_model.fit(
        X_unet, Y_unet, 
        epochs=30, # Ajusta las épocas según los resultados que observes
        batch_size=4,
        validation_split=0.2,
        verbose=1
    )
    print("Entrenamiento de U-Net finalizado.")

In [ ]:
def extraer_caracteristicas(mascara_bin, grid=(9, 6)):
    caracteristicas = []
    h, w = mascara_bin.shape
    cell_h, cell_w = h // grid[0], w // grid[1]
    
    for i in range(grid[0]):
        for j in range(grid[1]):
            cuadrante = mascara_bin[i*cell_h:(i+1)*cell_h, j*cell_w:(j+1)*cell_w]
            densidad = np.sum(cuadrante) / (cell_h * cell_w)
            caracteristicas.append(densidad)
    return np.array(caracteristicas)

if len(X_unet) > 0:
    print("Aplicando U-Net para segmentar el dataset completo...")
    mascaras_predichas = unet_model.predict(X_unet, verbose=0)
    
    X_features = []
    for mascara in mascaras_predichas:
        mascara_bin = (mascara > 0.5).astype(np.uint8).reshape(IMG_SIZE, IMG_SIZE)
        features = extraer_caracteristicas(mascara_bin)
        X_features.append(features)
        
    X_features = np.array(X_features)
    print(f"Matriz de características (Grid 9x6) obtenida: {X_features.shape}")

In [ ]:
if len(X_features) > 0:
    # Dividimos los datos para los clasificadores
    X_train, X_test, y_train, y_test = train_test_split(X_features, y_target, test_size=0.2, random_state=42, stratify=y_target)

    # 1. Backpropagation Neural Network (NNBP)
    nnbp = MLPClassifier(hidden_layer_sizes=(30,), activation='logistic', max_iter=2000, random_state=42)
    nnbp.fit(X_train, y_train)

    # 2. Support Vector Machine (SVM)
    svm = SVC(kernel='rbf', random_state=42)
    svm.fit(X_train, y_train)

    # 3. K-Nearest Neighbors (KNN)
    knn = KNeighborsClassifier(n_neighbors=3)
    knn.fit(X_train, y_train)

    # 4. Naive Bayes (NB)
    nb = GaussianNB()
    nb.fit(X_train, y_train)

    print("Clasificadores finales (Fase 4) entrenados desde cero exitosamente.")

In [ ]:
def pipeline_prediccion_completo(img_path):
    if not os.path.exists(img_path):
        return f"Archivo no encontrado: {img_path}"
        
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img_norm = img_resized / 255.0
    img_input = img_norm.reshape(1, IMG_SIZE, IMG_SIZE, 1)
    
    # 1. Segmentar con la U-Net entrenada desde cero
    mascara_pred = unet_model.predict(img_input, verbose=0)[0]
    mascara_bin = (mascara_pred > 0.5).astype(np.uint8).reshape(IMG_SIZE, IMG_SIZE)
    
    # 2. Extraer características
    features = extraer_caracteristicas(mascara_bin).reshape(1, -1)
    
    # 3. Clasificar con NNBP
    pred = nnbp.predict(features)[0]
    
    return "Auténtica (+5)" if pred == 5 else "Falsificación (-5)"

print("=== Resultados de Prueba Pipeline ===")
print("Prueba Real (esperado +5):", pipeline_prediccion_completo(TEST_REAL_PATH))
print("Prueba Fake (esperado -5):", pipeline_prediccion_completo(TEST_FAKE_PATH))